In [2]:
!pip install -q --upgrade langchain langchain-core langchain-community pypdf sentence_transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 437.6/437.6 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.3/302.3 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.7/345.7 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56

In [3]:
!pip install -qU "langchain[groq]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.5/127.5 kB 3.2 MB/s eta 0:00:00


In [4]:
import langchain
print(langchain.__version__)

0.3.25


# Simple LLM conversation with help of prompt

## Call LLM

In [5]:
import getpass
import os

# Check if the environment variable for the GROQ API key is already set
if not os.environ.get("GROQ_API_KEY"):
    # Prompt the user to enter the API key securely, and set it as an environment variable
  os.environ["GROQ_API_KEY"] = getpass.getpass("Enter API key for Groq: ")

from langchain.chat_models import init_chat_model

# Initialize the chat model using Groq as the provider and LLaMA3-8B as the selected model
model = init_chat_model("llama3-8b-8192", model_provider="groq")

Enter API key for Groq: ··········


## Use the initialized LLM to respond to a simple prompt

In [6]:
model_response=model.invoke("Tell me a joke")
model_response

AIMessage(content="Here's one:\n\nWhy couldn't the bicycle stand up by itself?\n\nBecause it was two-tired!\n\nHope that made you laugh!", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 14, 'total_tokens': 42, 'completion_time': 0.023333333, 'prompt_time': 0.002257903, 'queue_time': 0.06705107499999999, 'total_time': 0.025591236}, 'model_name': 'llama3-8b-8192', 'system_fingerprint': 'fp_179b0f92c9', 'finish_reason': 'stop', 'logprobs': None}, id='run--782c6edd-e25b-423f-a892-97373ab2b95a-0', usage_metadata={'input_tokens': 14, 'output_tokens': 28, 'total_tokens': 42})

## Parsing Output

In [7]:
from langchain_core.output_parsers import StrOutputParser

# Initialize the output parser that extracts just the plain text
output_parser=StrOutputParser()

# Parse and extract the plain text from the model's response
output_parser.invoke(model_response)

"Here's one:\n\nWhy couldn't the bicycle stand up by itself?\n\nBecause it was two-tired!\n\nHope that made you laugh!"

## Simple Chain

In [8]:
# Create a chain that sends input to the model and extracts the plain string output
chain=model | output_parser

# Invoke the chain with a prompt; this will return the cleaned text response
chain.invoke("Tell me a joke")

"Here's one:\n\nWhy couldn't the bicycle stand up by itself?\n\n(Wait for it...)\n\nBecause it was two-tired!\n\nHope that made you smile!"

## Structured Output

In [9]:
from typing import List
from pydantic import BaseModel, Field


# Define the structure of the expected output using a Pydantic model
class MobileReview(BaseModel):
  phone_model: str=Field(description="Name and model of the phone")
  rating: float= Field(description="Overall rating out of 5")
  pros: List[str]=Field(description="List of positive aspects")
  cons: List[str]=Field(description="List of negative aspects")
  summary: str=Field(description="Brief summary of the review")



# Example unstructured review text for parsing
review_text="""
Just got my hands on the new Galaxy S21 and wow, this thing is slick| The screen is gorgous,
colors pop like crazy. Camera's insane too, especially at night-my Insta game's never been stronger. Battery life's solid, lasts me all day no problem.


Not gonna lie though, it's pretty pricey. And what's with ditching the charger?C'mon Samsung.
Also, still getting used to the new button layout, keep hitting Bixby by mistake.


Overall, I'd say it's a solid 4 out of 5. Great phone, but a few annoying quirks keep it from
being perfect. If you're due for an upgrade, definitely worth checking out|"""




# Tell the model to return its output in the defined MobileReview format
structured_llm=model.with_structured_output(MobileReview)



# Invoke the model and return the structured result
output=structured_llm.invoke(review_text)
output

MobileReview(phone_model='Galaxy S21', rating=4.0, pros=['gorgeous screen', 'colors pop like crazy', 'insane camera, especially at night'], cons=['pricey', 'ditching the charger', 'new button layout takes getting used to'], summary='Solid phone, but a few annoying quirks keep it from being perfect')

In [10]:
output.phone_model

'Galaxy S21'

In [11]:
output.rating

4.0

In [12]:
output.pros

['gorgeous screen',
 'colors pop like crazy',
 'insane camera, especially at night']

In [13]:
output.cons

['pricey', 'ditching the charger', 'new button layout takes getting used to']

In [14]:
output.summary

'Solid phone, but a few annoying quirks keep it from being perfect'

## Prompt Template

In [15]:
# Import the class to define chat-based prompt templates
from langchain_core.prompts import ChatPromptTemplate


# Create a reusable prompt with a placeholder for a topic
prompt=ChatPromptTemplate.from_template("Tell me a short joke about {topic}")


# Fill in the topic to generate the full prompt
prompt.invoke({"topic":"programming"})

ChatPromptValue(messages=[HumanMessage(content='Tell me a short joke about programming', additional_kwargs={}, response_metadata={})])

In [17]:
# Create a LangChain pipeline to generate and parse a joke about a given topic
# Step 1: Format prompt with topic using the template
# Step 2: Pass prompt to the LLM model
# Step 3: Parse the output into plain text
chain=prompt|model|output_parser


# Invoke the pipeline with a specific topic
chain.invoke({"topic":"car drivers"})

'Why did the car driver bring a ladder to the road?\n\nBecause they wanted to take their driving to the next level!'

## LLM Messages

In [18]:
# Import message types for structured chat interaction
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, SystemMessage


# Define the assistant's behavior
system_message=SystemMessage(content="You are a helpful assistant that tells jokes.")


# User prompt to the assistant
human_message=HumanMessage(content="Tell me about programming")

# Send both messages to the model and get a response
model.invoke([system_message, human_message])

AIMessage(content='Programming! It\'s like trying to solve a puzzle blindfolded while being attacked by a swarm of bees... but in a good way!\n\nYou know, programming is like writing a recipe for a computer. You take some ingredients (like variables and functions), mix them together in a special order, and voilà! You get a delicious program that can do all sorts of useful things.\n\nBut don\'t worry if it gets a little messy – that\'s what debugging is for! It\'s like trying to find the one missing ingredient in your recipe. You test, you iterate, and you refine until it\'s just right.\n\nAnd when you finally get it working, it\'s like... MAGIC! Well, maybe not magic, but it\'s definitely a sense of accomplishment. You get to say, "Hey, I wrote this thing! It does stuff!"\n\nBut programming isn\'t just about writing code; it\'s also about problem-solving, critical thinking, and creativity. It\'s like being a detective, trying to figure out what\'s going on and how to fix it. And when y

In [19]:
# Create a dynamic chat prompt template with placeholders
template= ChatPromptTemplate([
    ("system","You are a helpful assistant that tells jokes."),
    ("human", "Tell me about {topic}")

])


# Invoke the template, replacing the placeholder with the value "programming"
prompt_value=template.invoke(
    {
        "topic":"programming"
    }
)

# The generated prompt
prompt_value

ChatPromptValue(messages=[SystemMessage(content='You are a helpful assistant that tells jokes.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Tell me about programming', additional_kwargs={}, response_metadata={})])

In [20]:
# Pass the formatted prompt to the model to get a response
model.invoke(prompt_value)

AIMessage(content="Programming! It's like trying to solve a puzzle blindfolded while being chased by a pack of wild algorithms!\n\nBut seriously, programming is like building with digital Legos. You create blocks of code, and with the right combination, you can create something amazing! Like a robot that can make you a sandwich, or a website that can guess your favorite joke (just kidding, I'm on a roll with these jokes).\n\nDid you hear about the programmer who quit his job because he didn't get arrays? (get it? arrays... he didn't get arrays... ahh, nevermind!)\n\nAnyway, programming is all about solving problems, and when you finally figure out that tricky bug, it's like winning a digital gold medal!\n\nWant to hear another one? Why did the programmer go to the doctor? Because he was feeling a little glitchy!\n\nOkay, okay, one more. Why did the programmer get lost in the forest? Because he was too busy debugging his map! (ba-dum-tss)\n\nI hope those made you LOL and not feel like y

# Simple RAG Creation

## Loading documents

In [21]:
!pip install -qU langchain-huggingface

In [22]:
# Import necessary modules for document loading and processing
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document


# Define a text splitter to divide the document into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # Maximum size of each chunk in characters
    chunk_overlap=200, # Overlap between consecutive chunks to maintain context
    length_function=len # Length function to count characters in each chunk
)


# Load the PDF document from the specified path
pdf_loader = PyPDFLoader("/content/Apple_Environmental_Progress_Report_2024.pdf")
documents = pdf_loader.load() # Read the content of the PDF into a list of documents



# Split the loaded documents into smaller chunks
splits = text_splitter.split_documents(documents)


# Print the number of chunks created after splitting
print(f"Split the documents into {len(splits)} chunks.")

Split the documents into 489 chunks.


In [23]:
splits[0]

Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.3 (Macintosh)', 'creationdate': '2024-05-15T13:41:41-07:00', 'author': 'Apple, Inc.', 'keywords': 'Annual environmental report covering fiscal year 2023.', 'moddate': '2024-05-15T13:54:52-07:00', 'title': '2024 Environmental Progress Report', 'trapped': '/False', 'source': '/content/Apple_Environmental_Progress_Report_2024.pdf', 'total_pages': 113, 'page': 0, 'page_label': '1'}, page_content='Environmental \nProgress \nReport\nCovering fiscal year 2023')

In [24]:
splits[0].metadata

{'producer': 'Adobe PDF Library 17.0',
 'creator': 'Adobe InDesign 19.3 (Macintosh)',
 'creationdate': '2024-05-15T13:41:41-07:00',
 'author': 'Apple, Inc.',
 'keywords': 'Annual environmental report covering fiscal year 2023.',
 'moddate': '2024-05-15T13:54:52-07:00',
 'title': '2024 Environmental Progress Report',
 'trapped': '/False',
 'source': '/content/Apple_Environmental_Progress_Report_2024.pdf',
 'total_pages': 113,
 'page': 0,
 'page_label': '1'}

In [25]:
splits[0].page_content

'Environmental \nProgress \nReport\nCovering fiscal year 2023'

In [29]:
# Initialize the HuggingFace embeddings model with a pre-trained model
embeddings_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")



# Embed the document chunks (converted to text) into embedding vectors
document_embeddings=embeddings_model.embed_documents([split.page_content for split in splits])


# Print the first embedding vector to verify the embeddings are generated correctly
print(f"First embedding vector: {document_embeddings[0]}")

First embedding vector: [0.008061584085226059, 0.1471094787120819, -0.01207153219729662, 0.011529283598065376, -0.014869561418890953, -0.030456513166427612, -0.029251903295516968, 0.01306227408349514, -0.04191932827234268, -0.016437934711575508, 0.016473717987537384, 0.06640420854091644, 0.03222237527370453, 0.062331970781087875, 0.01907031424343586, -0.08827867358922958, 0.05417870730161667, -0.0334915928542614, -0.0659298449754715, -0.013331080786883831, -0.017764970660209656, -0.01425209641456604, -0.0006309827440418303, 0.003951416350901127, 0.013116145506501198, 0.025569748133420944, -0.039580658078193665, 0.045560769736766815, -0.04138600081205368, -0.03802129998803139, 0.06171732023358345, -0.003300819080322981, 0.045184675604104996, -0.05603699013590813, 1.7400377601006767e-06, -0.03309507668018341, -0.025746900588274002, -0.003536204108968377, 0.01724264584481716, 0.035439714789390564, 0.06588118523359299, -0.04695800691843033, -0.029103314504027367, 0.024485798552632332, 0.04

In [30]:
!pip install faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 24.0 MB/s eta 0:00:00


## Create vector store

In [32]:
from langchain_community.vectorstores import FAISS

# Create a FAISS vector store by embedding the document chunks using the pre-trained embedding model
vectorstore = FAISS.from_documents(splits, embeddings_model)

In [33]:
# Define the file path where the FAISS index will be saved
faiss_index_path = "/content/faiss_index"



# Save the FAISS index to the local file system so that it can be loaded later
vectorstore.save_local(faiss_index_path)

# Print confirmation message
print(f"FAISS index saved to {faiss_index_path}")

FAISS index saved to /content/faiss_index


In [35]:
# Load the FAISS index from the saved file, providing the embeddings model and allowing deserialization
vectorstore_loaded = FAISS.load_local(
    folder_path=faiss_index_path,  # Path where the index was saved
    embeddings=embeddings_model,    # The embedding model used for the index
    allow_dangerous_deserialization=True  # Allow deserialization (ensure trust in the source)
)

# Print a success message when the FAISS index is loaded
print("FAISS index loaded successfully.")


FAISS index loaded successfully.


In [36]:
# Define the query to search for relevant information in the FAISS index
query = "What are Apple's 2023 environmental goals?"

# Perform a similarity search on the FAISS index and retrieve the top 2 most relevant documents
search_results = vectorstore.similarity_search(query, k=2)

# Print the top 2 results for the query
print(f"\nTop 2 most relevant chunks for the query: '{query}'\n")

# Loop through the search results and print each one with its source and content
for i, result in enumerate(search_results, 1):
    print(f"Result {i}:")
    print(f"Source: {result.metadata.get('source', 'Unknown')}")
    print(f"Content: {result.page_content}")
    print()



Top 2 most relevant chunks for the query: 'What are Apple's 2023 environmental goals?'

Result 1:
Source: /content/Apple_Environmental_Progress_Report_2024.pdf
Content: Approach
Apple 2030
We have an ambitious commitment 
and a science-based plan to reach 
our Apple 2030 goal. We’re focused 
on achieving reductions wherever 
possible, using approaches that 
offer clear evidence for a way 
forward while seeking to catalyze 
industry-wide change.
This begins with working to achieve carbon 
neutrality across our entire carbon footprint by 2030, 
setting ambitious targets to reduce our emissions 
by 75 percent. We prioritize carbon reductions, 
but for emissions that can’t be mitigated using 
existing solutions we invest in high-quality carbon 
removal projects.
Our goal to be carbon neutral extends to our 
entire carbon footprint and is consistent with the 
Intergovernmental Panel on Climate Change’s (IPCC) 
recommendation for global carbon neutrality.24 
We’re also committed to working 

In [37]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
retriever.invoke("What was Apple's total environmental spend in 2023?")

[Document(id='6f36f127-6518-448b-8098-892bfd20a36a', metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.3 (Macintosh)', 'creationdate': '2024-05-15T13:41:41-07:00', 'author': 'Apple, Inc.', 'keywords': 'Annual environmental report covering fiscal year 2023.', 'moddate': '2024-05-15T13:54:52-07:00', 'title': '2024 Environmental Progress Report', 'trapped': '/False', 'source': '/content/Apple_Environmental_Progress_Report_2024.pdf', 'total_pages': 113, 'page': 82, 'page_label': '83'}, page_content='Data\nNormalizing factors*\nFiscal year\n2023 2022 2021 2020 2019\nNet sales (in millions, US$) 383,285 394,328 365,817 274,515 260,174\nNumber of full-time equivalent employees 161,000 164,000 154,000 147,000 137,000\n* As reported in Apple’s Form 10-K Annual Report \nfiled with the SEC.\n2024  Environmental Progress Report  83Engagement and AdvocacyEnvironmental Initiatives AppendixIntroduction Contents Data'),
 Document(id='dc2f8596-8cb4-427e-a416-0ce8ca3622a6', m

In [38]:
from langchain_core.prompts import ChatPromptTemplate
template=""" Answer the question based only on the following context:
{context}

Question:{question}

Answer:"""

prompt=ChatPromptTemplate.from_template(template)

In [39]:
from langchain.schema.runnable import RunnablePassthrough

# Assuming `retriever` is your FAISS retriever and `prompt` is your ChatPromptTemplate
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()} | prompt
)

# Now, invoke the chain with your question
rag_chain.invoke("What are Apple's 2023 environmental goals?")




ChatPromptValue(messages=[HumanMessage(content=" Answer the question based only on the following context:\n[Document(id='1e271163-c5e6-4af3-b4e5-c24eeeac2047', metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.3 (Macintosh)', 'creationdate': '2024-05-15T13:41:41-07:00', 'author': 'Apple, Inc.', 'keywords': 'Annual environmental report covering fiscal year 2023.', 'moddate': '2024-05-15T13:54:52-07:00', 'title': '2024 Environmental Progress Report', 'trapped': '/False', 'source': '/content/Apple_Environmental_Progress_Report_2024.pdf', 'total_pages': 113, 'page': 11, 'page_label': '12'}, page_content='Approach\\nApple 2030\\nWe have an ambitious commitment \\nand a science-based plan to reach \\nour Apple 2030 goal. We’re focused \\non achieving reductions wherever \\npossible, using approaches that \\noffer clear evidence for a way \\nforward while seeking to catalyze \\nindustry-wide change.\\nThis begins with working to achieve carbon \\nneutrality across 

In [40]:
def doc2str(docs):
  return "\n\n".join(doc.page_content for doc in docs)

In [41]:
rag_chain=(
    {"context": retriever | doc2str, "question": RunnablePassthrough() } | prompt
)

rag_chain.invoke("What are Apple's 2023 environmental goals?")

ChatPromptValue(messages=[HumanMessage(content=" Answer the question based only on the following context:\nApproach\nApple 2030\nWe have an ambitious commitment \nand a science-based plan to reach \nour Apple 2030 goal. We’re focused \non achieving reductions wherever \npossible, using approaches that \noffer clear evidence for a way \nforward while seeking to catalyze \nindustry-wide change.\nThis begins with working to achieve carbon \nneutrality across our entire carbon footprint by 2030, \nsetting ambitious targets to reduce our emissions \nby 75 percent. We prioritize carbon reductions, \nbut for emissions that can’t be mitigated using \nexisting solutions we invest in high-quality carbon \nremoval projects.\nOur goal to be carbon neutral extends to our \nentire carbon footprint and is consistent with the \nIntergovernmental Panel on Climate Change’s (IPCC) \nrecommendation for global carbon neutrality.24\xa0\nWe’re also committed to working toward reaching \na 90 percent reductio

In [42]:
rag_chain=(
    {"context": retriever| doc2str, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

question= "What are Apple's 2023 environmental goals?"
response= rag_chain.invoke(question)
print(response)

There is no mention of Apple's 2023 environmental goals in the provided context. The text only mentions Apple's goals for 2030 and 2050, but not 2023.


# Conversational RAG

Handling Follow up questions

In [43]:
# Example conversation
from langchain_core.messages import HumanMessage, AIMessage
chat_history=[]
chat_history.extend([
    HumanMessage(content=question),
    AIMessage(content=response)
])

In [44]:
chat_history

[HumanMessage(content="What are Apple's 2023 environmental goals?", additional_kwargs={}, response_metadata={}),
 AIMessage(content="There is no mention of Apple's 2023 environmental goals in the provided context. The text only mentions Apple's goals for 2030 and 2050, but not 2023.", additional_kwargs={}, response_metadata={})]

In [45]:
from langchain_core.prompts import MessagesPlaceholder
from langchain.chains import create_history_aware_retriever
from langchain.chains.combine_documents import create_stuff_documents_chain

contextualize_q_system_prompt = """
Given a chat history and the latest user question
which might reference context in the chat history,
formulate a standalone question which can be understood
without the chat history. Do NOT answer the question,
just reformulate it if needed and otherwise return it as is.
"""

contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

contextualize_chain = contextualize_q_prompt | model | StrOutputParser()
print(contextualize_chain.invoke({"input": "What financial incentives did Apple receive for sustainability efforts?", "chat_history": []}))

What incentives did Apple receive for its sustainability efforts?


In [46]:
from langchain.chains import create_history_aware_retriever
history_aware_retriever=create_history_aware_retriever(
    model, retriever, contextualize_q_prompt
)
history_aware_retriever.invoke({"input": "What financial incentives did Apple receive for sustainability efforts?", "chat_history": []})

[Document(id='0eef375f-9c0f-41ad-9d2b-7b99c28aea28', metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.3 (Macintosh)', 'creationdate': '2024-05-15T13:41:41-07:00', 'author': 'Apple, Inc.', 'keywords': 'Annual environmental report covering fiscal year 2023.', 'moddate': '2024-05-15T13:54:52-07:00', 'title': '2024 Environmental Progress Report', 'trapped': '/False', 'source': '/content/Apple_Environmental_Progress_Report_2024.pdf', 'total_pages': 113, 'page': 111, 'page_label': '112'}, page_content='End\xa0notes\nIntroduction Environmental Initiatives\n1  Apple follows the GHG Protocol Corporate Accounting and \nReporting Standard (GHG Protocol) to calculate value chain \nemissions. The GHG Protocol currently defines scope 1 emissions \nas direct greenhouse gas emissions that occur from sources that \nare owned or controlled by the company; scope 2 emissions as \nthe indirect greenhouse gas emissions from the generation of \npurchased electricity, steam, heat,

In [48]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant. Use the following context to answer the user's question."),
    ("system", "Context: {context}"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

question_answer_chain=create_stuff_documents_chain(model, qa_prompt)

rag_chain=create_retrieval_chain(history_aware_retriever, question_answer_chain)



In [49]:
rag_chain.invoke({"input": "What financial incentives did Apple receive for sustainability efforts?", "chat_history": []})

{'input': 'What financial incentives did Apple receive for sustainability efforts?',
 'chat_history': [],
 'context': [Document(id='0eef375f-9c0f-41ad-9d2b-7b99c28aea28', metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.3 (Macintosh)', 'creationdate': '2024-05-15T13:41:41-07:00', 'author': 'Apple, Inc.', 'keywords': 'Annual environmental report covering fiscal year 2023.', 'moddate': '2024-05-15T13:54:52-07:00', 'title': '2024 Environmental Progress Report', 'trapped': '/False', 'source': '/content/Apple_Environmental_Progress_Report_2024.pdf', 'total_pages': 113, 'page': 111, 'page_label': '112'}, page_content='End\xa0notes\nIntroduction Environmental Initiatives\n1  Apple follows the GHG Protocol Corporate Accounting and \nReporting Standard (GHG Protocol) to calculate value chain \nemissions. The GHG Protocol currently defines scope 1 emissions \nas direct greenhouse gas emissions that occur from sources that \nare owned or controlled by the company; sco

# Building Multi User Chatbot

In [50]:
import sqlite3
from datetime import datetime
import uuid

DB_NAME = "rag_app.db"

def get_db_connection():
    conn = sqlite3.connect(DB_NAME)
    conn.row_factory = sqlite3.Row
    return conn

def create_application_logs():
    conn = get_db_connection()
    conn.execute('''CREATE TABLE IF NOT EXISTS application_logs
    (id INTEGER PRIMARY KEY AUTOINCREMENT,
    session_id TEXT,
    user_query TEXT,
    gpt_response TEXT,
    model TEXT,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)''')
    conn.close()

def insert_application_logs(session_id, user_query, gpt_response, model):
    conn = get_db_connection()
    conn.execute('INSERT INTO application_logs (session_id, user_query, gpt_response, model) VALUES (?, ?, ?, ?)',
                 (session_id, user_query, gpt_response, model))
    conn.commit()
    conn.close()

def get_chat_history(session_id):
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute('SELECT user_query, gpt_response FROM application_logs WHERE session_id = ? ORDER BY created_at', (session_id,))
    messages = []
    for row in cursor.fetchall():
        messages.extend([
            {"role": "human", "content": row['user_query']},
            {"role": "ai", "content": row['gpt_response']}
        ])
    conn.close()
    return messages

# Initialize the database
create_application_logs()


In [52]:
# Example usage for a new user
session_id = str(uuid.uuid4())
question = "What portion of Apple's budget was dedicated to green initiatives in 2023?"
chat_history = get_chat_history(session_id)
print(chat_history)
answer = rag_chain.invoke({"input": question, "chat_history": chat_history})['answer']
insert_application_logs(session_id, question, answer, "gpt-3.5-turbo")
print(f"Human: {question}")
print(f"AI: {answer}\n")




[]
Human: What portion of Apple's budget was dedicated to green initiatives in 2023?
AI: Based on the provided data, we can estimate the portion of Apple's budget dedicated to green initiatives in 2023.

The data shows that Apple's net sales in 2023 were $383,285 million. However, there is no direct information on the budget dedicated to green initiatives.

To make an estimate, we can look at Apple's environmental progress report, which mentions that the company has a goal to be carbon neutral for its entire footprint. Apple has made significant progress towards this goal, with a reduction of 16.1 million metric tons of CO2e emissions in 2023.

While we don't have a direct figure on the budget dedicated to green initiatives, we can make an educated estimate based on Apple's commitment to environmental sustainability. Given the company's focus on reducing emissions and achieving its carbon neutrality goal, it's likely that a significant portion of its budget is dedicated to green initia

In [53]:
# Example of a follow-up question
question2 = "How did Apple finance its renewable energy projects in 2023?"
chat_history = get_chat_history(session_id)
print(chat_history)
answer2 = rag_chain.invoke({"input": question2, "chat_history": chat_history})['answer']
insert_application_logs(session_id, question2, answer2, "gpt-3.5-turbo")
print(f"Human: {question2}")
print(f"AI: {answer2}")

[{'role': 'human', 'content': "What portion of Apple's budget was dedicated to green initiatives in 2023?"}, {'role': 'ai', 'content': "Based on the provided data, we can estimate the portion of Apple's budget dedicated to green initiatives in 2023.\n\nThe data shows that Apple's net sales in 2023 were $383,285 million. However, there is no direct information on the budget dedicated to green initiatives.\n\nTo make an estimate, we can look at Apple's environmental progress report, which mentions that the company has a goal to be carbon neutral for its entire footprint. Apple has made significant progress towards this goal, with a reduction of 16.1 million metric tons of CO2e emissions in 2023.\n\nWhile we don't have a direct figure on the budget dedicated to green initiatives, we can make an educated estimate based on Apple's commitment to environmental sustainability. Given the company's focus on reducing emissions and achieving its carbon neutrality goal, it's likely that a significa

# New User

In [55]:
session_id=str(uuid.uuid4())
question="What was the return on investment (ROI) of Apple’s green infrastructure projects?"
chat_history=get_chat_history(session_id)
print(chat_history)
answer=rag_chain.invoke({"input": question, "chat_history": chat_history})['answer']
insert_application_logs(session_id, question, answer, "gpt-3.5-turbo")
print(f"Human: {question}")
print(f"AI: {answer}\n")

[]
Human: What was the return on investment (ROI) of Apple’s green infrastructure projects?
AI: The context does not provide information on the return on investment (ROI) of Apple's green infrastructure projects. The provided text only mentions the investment approaches Apple used to create new renewable energy projects, which include direct ownership and equity investment. It also mentions the total amount of committed capital for the Restore Fund, but it does not provide information on the financial returns or ROI of these projects.

However, it's worth noting that investing in renewable energy and green infrastructure can provide long-term benefits, such as reduced energy costs, improved brand reputation, and compliance with environmental regulations. These benefits can contribute to a positive ROI, but the specific return on investment for Apple's projects is not publicly disclosed.

